# Notebook 2 
## Benign Transaction Collection
### This notebook focuses on gathering transactions from reputable Ethereum addresses.

In [1]:
# %% [markdown]
# # Notebook 2: Benign Transaction Collection
# This notebook focuses on gathering transactions from reputable Ethereum addresses
# to create a dataset of benign (non-malicious) transactions.

# %%
# ----- CODE CELL: Imports and Configuration -----
from pathlib import Path
import json
from etherscan import Etherscan # Import Etherscan
import os
from dotenv import load_dotenv # Import load_dotenv
import time

# Load Etherscan API key from .env file
load_dotenv() 
ETH_CLIENT = Etherscan(os.getenv("ETHERSCAN_API_KEY")) # Use a distinct name like ETH_CLIENT

# Define list of benign addresses to fetch transactions for
# These are examples; you should replace them with your researched addresses.
BENIGN_ADDRESSES_TO_COLLECT = {
    "WETH_Contract": "0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2",
    "USDC_Contract": "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48",
    "UniswapV2_Router": "0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D",
    "DAI_Stablecoin": "0x6B175474E89094C44Da98b954EedeAC495271d0F",
    "Compound_cDAI": "0x5d3a536E4D6DbD6114cc1Ead35777bAB948E3643",
}

# Configure paths assuming this notebook is in 'EthMalViz/notebooks/'
PROJECT_ROOT_FROM_NOTEBOOKS_NB02 = Path('..') 
RAW_BENIGN_TX_DIR = PROJECT_ROOT_FROM_NOTEBOOKS_NB02 / 'data' / 'raw' / 'benign_transactions'
RAW_BENIGN_TX_DIR.mkdir(parents=True, exist_ok=True) # Ensure the directory exists

# Limit the number of transactions to fetch per address
MAX_TRANSACTIONS_PER_ADDRESS_NB02 = 5000 

print("Benign addresses selected for collection:")
for name, addr in BENIGN_ADDRESSES_TO_COLLECT.items():
    print(f"  - {name}: {addr}")
print(f"\nRaw benign transactions will be saved to: {RAW_BENIGN_TX_DIR}")

# %%
# ----- CODE CELL: Transaction Fetching Logic -----
total_benign_tx_fetched_count_nb02 = 0
print("\nStarting benign transaction fetching process...")

for entity_name, address_hex in BENIGN_ADDRESSES_TO_COLLECT.items():
    print(f"\nFetching transactions for: {entity_name} ({address_hex})...")
    output_file_path = RAW_BENIGN_TX_DIR / f"{address_hex}_benign.json" # Unique filename
    
    if output_file_path.exists():
        print(f"  Data for {address_hex} already exists at {output_file_path}. Skipping.")
        continue

    benign_tx_list_for_address = []
    try:
        fetched_tx = ETH_CLIENT.get_normal_txs_by_address( # Use ETH_CLIENT
            address=address_hex,
            startblock=0,
            endblock=99999999,
            sort="desc"
        )
        
        if fetched_tx:
            benign_tx_list_for_address = fetched_tx[:MAX_TRANSACTIONS_PER_ADDRESS_NB02]
        else:
            benign_tx_list_for_address = []

        if benign_tx_list_for_address:
            with open(output_file_path, 'w') as f_out_benign:
                json.dump(benign_tx_list_for_address, f_out_benign, indent=4)
            print(
                f"  Successfully fetched and saved {len(benign_tx_list_for_address)} "
                f"transactions to {output_file_path.name}" # Use .name for just filename
            )
            total_benign_tx_fetched_count_nb02 += len(benign_tx_list_for_address)
        else:
            print(f"  No transactions found or returned for {address_hex}.")
            
    except Exception as e:
        print(f"  ERROR fetching transactions for {address_hex}: {e}")
    
    time.sleep(0.21) # Respect API rate limits

print(
    f"\nBenign transaction fetching complete. "
    f"Total new benign transactions fetched in this run: "
    f"{total_benign_tx_fetched_count_nb02}"
)


Benign addresses selected for collection:
  - WETH_Contract: 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2
  - USDC_Contract: 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
  - UniswapV2_Router: 0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D
  - DAI_Stablecoin: 0x6B175474E89094C44Da98b954EedeAC495271d0F
  - Compound_cDAI: 0x5d3a536E4D6DbD6114cc1Ead35777bAB948E3643

Raw benign transactions will be saved to: ../data/raw/benign_transactions

Starting benign transaction fetching process...

Fetching transactions for: WETH_Contract (0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2)...
  Data for 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2 already exists at ../data/raw/benign_transactions/0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2_benign.json. Skipping.

Fetching transactions for: USDC_Contract (0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48)...
  Data for 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48 already exists at ../data/raw/benign_transactions/0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48_benign.json. Skipping.
